## Week 9: Gradient Boost

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42

### Dataset 1: Animation Movies 

In [3]:
df1_animation = pd.read_csv("data/Animation_Movies_clean.csv")

In [4]:
df1_animation["budget"] = df1_animation["budget"].replace(0, np.nan)
df1_animation["revenue"] = df1_animation["revenue"].replace(0, np.nan)

# Create log-transformed variables
df1_animation["log_budget"] = np.log10(df1_animation["budget"])
df1_animation["log_revenue"] = np.log10(df1_animation["revenue"])
df1_animation["log_popularity"] = np.log10(
    df1_animation["popularity"].where(df1_animation["popularity"] > 0)
)


In [6]:
animation_features = [
    "log_budget",
    "log_popularity",
    "vote_average",
    "vote_count",
    "runtime"
]

animation_week9 = df1_animation.dropna(
    subset=animation_features + ["log_revenue"]
).copy()

animation_week9["high_revenue"] = (
    animation_week9["log_revenue"] >=
    animation_week9["log_revenue"].median()
).astype(int)

In [8]:
print("Animation sample:", animation_week9.shape)


print("\nAnimation class distribution:")
print(animation_week9["high_revenue"].value_counts())



Animation sample: (616, 24)

Animation class distribution:
high_revenue
1    308
0    308
Name: count, dtype: int64


In [12]:
X_animation = animation_week9[animation_features]
y_animation = animation_week9["high_revenue"]

X_train_anim, X_test_anim, y_train_anim, y_test_anim = train_test_split(
    X_animation,
    y_animation,
    test_size=0.2,
    random_state=42,
    stratify=y_animation
)


In [15]:
gb_animation = GradientBoostingClassifier(
    learning_rate=0.1,
    n_estimators=100,
    max_depth=3,
    random_state=42
)

# Train model
gb_animation.fit(X_train_anim, y_train_anim)

# Predictions
y_pred_anim = gb_animation.predict(X_test_anim)

# Evaluate
print("Animation Gradient Boosting Results")
print("Training Accuracy:", gb_animation.score(X_train_anim, y_train_anim))
print("Testing Accuracy:", accuracy_score(y_test_anim, y_pred_anim))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_anim, y_pred_anim))

print("\nClassification Report:")
print(classification_report(y_test_anim, y_pred_anim))

Animation Gradient Boosting Results
Training Accuracy: 0.983739837398374
Testing Accuracy: 0.8629032258064516

Confusion Matrix:
[[51 11]
 [ 6 56]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.82      0.86        62
           1       0.84      0.90      0.87        62

    accuracy                           0.86       124
   macro avg       0.87      0.86      0.86       124
weighted avg       0.87      0.86      0.86       124



### Learning Rate

In [18]:
X_fit_anim, X_val_anim, y_fit_anim, y_val_anim = train_test_split(
    X_train_anim,
    y_train_anim,
    test_size=0.25,
    random_state=42,
    stratify=y_train_anim
)

In [19]:
learning_rates = [0.01, 0.05, 0.1, 0.2, 0.3]

animation_lr_results = []

for lr in learning_rates:
    model = GradientBoostingClassifier(
        learning_rate=lr,
        n_estimators=100,
        max_depth=3,
        random_state=42
    )

    model.fit(X_fit_anim, y_fit_anim)

    animation_lr_results.append({
        "Learning Rate": lr,
        "Training Accuracy": model.score(X_fit_anim, y_fit_anim),
        "Validation Accuracy": model.score(X_val_anim, y_val_anim)
    })

animation_lr_df = pd.DataFrame(animation_lr_results)

display(animation_lr_df.round(4))

,Learning Rate,Training Accuracy,Validation Accuracy
0,0.01,0.9187,0.8862
1,0.05,0.9702,0.8943
2,0.10,0.9973,0.8862
3,0.20,1.0000,0.8862
4,0.30,1.0000,0.8943


### n_estimators

In [21]:
n_estimators_values = [50, 100, 200, 300, 500]

animation_est_results = []

for n in n_estimators_values:
    model = GradientBoostingClassifier(
        n_estimators=n,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    model.fit(X_fit_anim, y_fit_anim)

    animation_est_results.append({
        "N Estimators": n,
        "Training Accuracy": model.score(X_fit_anim, y_fit_anim),
        "Validation Accuracy": model.score(X_val_anim, y_val_anim)
    })

animation_est_df = pd.DataFrame(animation_est_results)

display(animation_est_df.round(4))

,N Estimators,Training Accuracy,Validation Accuracy
0,50,0.9729,0.8943
1,100,0.9973,0.8862
2,200,1.0000,0.8862
3,300,1.0000,0.8862
4,500,1.0000,0.8699


### Dataset 2: Action Movies 

In [2]:
df2_action = pd.read_csv("data/Action_Movies_clean.csv")

In [5]:
# Create log-transformed variables
df2_action["log_votes"] = np.log10(
    df2_action["votes"].where(df2_action["votes"] > 0)
)

df2_action["log_gross"] = np.log10(
    df2_action["gross(in $)"].where(df2_action["gross(in $)"] > 0)
)


In [10]:
action_features = [
    "log_votes",
    "rating",
    "runtime",
    "year"
]

action_week9 = df2_action.dropna(
    subset=action_features + ["log_gross"]
).copy()

action_week9["high_gross"] = (
    action_week9["log_gross"] >=
    action_week9["log_gross"].median()
).astype(int)

In [9]:

print("Action sample:", action_week9.shape)


print("\nAction class distribution:")
print(action_week9["high_gross"].value_counts())

Action sample: (2758, 17)

Action class distribution:
high_gross
1    1379
0    1379
Name: count, dtype: int64


In [13]:
X_action = action_week9[action_features]
y_action = action_week9["high_gross"]

X_train_action, X_test_action, y_train_action, y_test_action = train_test_split(
    X_action,
    y_action,
    test_size=0.2,
    random_state=42,
    stratify=y_action
)

In [16]:
gb_action = GradientBoostingClassifier(
    learning_rate=0.1,
    n_estimators=100,
    max_depth=3,
    random_state=42
)

# Train model
gb_action.fit(X_train_action, y_train_action)

# Predictions
y_pred_action = gb_action.predict(X_test_action)

# Evaluate
print("Action Gradient Boosting Results")
print("Training Accuracy:", gb_action.score(X_train_action, y_train_action))
print("Testing Accuracy:", accuracy_score(y_test_action, y_pred_action))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_action, y_pred_action))

print("\nClassification Report:")
print(classification_report(y_test_action, y_pred_action))

Action Gradient Boosting Results
Training Accuracy: 0.9152311876699909
Testing Accuracy: 0.8913043478260869

Confusion Matrix:
[[239  37]
 [ 23 253]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.87      0.89       276
           1       0.87      0.92      0.89       276

    accuracy                           0.89       552
   macro avg       0.89      0.89      0.89       552
weighted avg       0.89      0.89      0.89       552



### Learning Rate

In [17]:
X_fit_action, X_val_action, y_fit_action, y_val_action = train_test_split(
    X_train_action,
    y_train_action,
    test_size=0.25,
    random_state=42,
    stratify=y_train_action
)

In [20]:
action_lr_results = []

for lr in learning_rates:
    model = GradientBoostingClassifier(
        learning_rate=lr,
        n_estimators=100,
        max_depth=3,
        random_state=42
    )

    model.fit(X_fit_action, y_fit_action)

    action_lr_results.append({
        "Learning Rate": lr,
        "Training Accuracy": model.score(X_fit_action, y_fit_action),
        "Validation Accuracy": model.score(X_val_action, y_val_action)
    })

action_lr_df = pd.DataFrame(action_lr_results)

display(action_lr_df.round(4))

,Learning Rate,Training Accuracy,Validation Accuracy
0,0.01,0.8748,0.8551
1,0.05,0.8996,0.8641
2,0.10,0.9196,0.8714
3,0.20,0.9541,0.8859
4,0.30,0.9740,0.8841


### n_estimators

In [22]:
action_est_results = []

for n in n_estimators_values:
    model = GradientBoostingClassifier(
        n_estimators=n,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    model.fit(X_fit_action, y_fit_action)

    action_est_results.append({
        "N Estimators": n,
        "Training Accuracy": model.score(X_fit_action, y_fit_action),
        "Validation Accuracy": model.score(X_val_action, y_val_action)
    })

action_est_df = pd.DataFrame(action_est_results)

display(action_est_df.round(4))

,N Estimators,Training Accuracy,Validation Accuracy
0,50,0.9002,0.8641
1,100,0.9196,0.8714
2,200,0.9486,0.8822
3,300,0.9710,0.8822
4,500,0.9897,0.8822
